In [1]:
from leakly import SimulationConfig, simulate_dataset

simulated = simulate_dataset(
    SimulationConfig(
        n_samples=1000,
        n_features=100,
        n_covariates=3,
        effect_fraction=0.1,
        effect_size=0.5,
        class_balance=0.2,
        random_state=42,
    )
)

In [2]:
from leakly import FeatureSelectionConfig, feature_selection

selected_features, selected_indices = feature_selection(
    simulated.X, 
    simulated.y, 
    simulated.covariates, 
    feature_names=simulated.feature_names,
    config=FeatureSelectionConfig(
        method="LinearRegressionDAA",
        alpha=0.05,
        correction_method="fdr_bh",
        minimum_effect_size=None,
        top_ranks=None,
    ),
)
print("Selected features:", selected_features)

LinearRegressionDAA:   0%|          | 0/100 [00:00<?, ?feature/s]

Selected features: ['feature_3', 'feature_8', 'feature_4', 'feature_7', 'feature_2', 'feature_1', 'feature_10', 'feature_6', 'feature_9', 'feature_5']


In [3]:
X, y, covariates = simulated.X, simulated.y, simulated.covariates
X = X[:, selected_indices]
print("Shape of selected feature matrix:", X.shape)

# randomly shuffle (X, y, covariates) for splitting train/test sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test, covariates_train, covariates_test = train_test_split(
    X, y, covariates, test_size=0.2, random_state=42
)

from leakly import ModelConfig, ml_model
test_auc = ml_model(
    X_train, 
    y_train, 
    X_test,
    y_test,
    config=ModelConfig(
        model="random_forest",
        problem_type="binary_classification",
        metric="auc",
        random_state=42,
        model_params={"n_estimators": 100, "max_depth": 5},
    ),
)

print("Test AUC:", test_auc)


Shape of selected feature matrix: (1000, 10)
Test AUC: 0.8333333333333333


In [5]:
from leakly import (
    MLPipeline,
    load_example_leakage_config,
    print_config,
)

config = load_example_leakage_config()

print_config(config)

pipeline = MLPipeline(
    simulated.X,
    simulated.y,
    covariates=simulated.covariates,
    config=config)
pipeline.fit()
pipeline.evaluate(metric="auc")

pipeline:
- imputation
- normalization
- feature_selection
- data_split
- model
imputation:
  method: knn
  n_neighbors: 5
normalization:
  method: zscore
  with_mean: true
  with_std: true
data_split:
  method: train_test
  test_fraction: 0.2
  random_state: null
  stratify: true
feature_selection:
  method: LinearRegressionDAA
  alpha: 0.05
  minimum_effect_size: 0.0
  top_ranks: 10
  correction_method: fdr_bh
  selected_feature_names: null
model:
  model: random_forest
  problem_type: binary_classification
  metric: auc
  random_state: 42
  model_params:
    n_estimators: 100
    max_depth: 5
    min_samples_split: 2
    min_samples_leaf: 1


LinearRegressionDAA:   0%|          | 0/100 [00:00<?, ?feature/s]

0.8546874999999999